In [1]:
import sys
print("python exe:", sys.executable)   # full path to the interpreter
print("python ver:", sys.version)

# try torch only in the working notebook
try:
    import torch
    print("torch      :", torch.__version__, torch.__file__)
except ModuleNotFoundError as e:
    print("torch not importable:", e)



python exe: c:\Users\aneek\anaconda3\envs\tf_gpu_env\python.exe
python ver: 3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:49:16) [MSC v.1929 64 bit (AMD64)]
torch      : 1.12.1+cu113 c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\torch\__init__.py


c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import tensorflow as tf
print(tf.__version__)  # This should print the version of TensorFlow
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

print("CUDA version:", tf.sysconfig.get_build_info()["cuda_version"])
print("cuDNN version:", tf.sysconfig.get_build_info()["cudnn_version"])

from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

2.10.0
Num GPUs Available:  1
CUDA version: 64_112
cuDNN version: 64_8
[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 14591249827311297855
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 5713690624
locality {
  bus_id: 1
  links {
  }
}
incarnation: 18130711483974970583
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9"
xla_global_id: 416903419
]


In [3]:
import time
import tensorflow as tf
import psutil

class PowerMonitor:
    def __init__(self):
        self.gpu_available = tf.config.list_physical_devices('GPU')
        
        # Hardware power specifications (adjust these values for your system)
        self.cpu_tdp = 65    # Typical TDP for desktop CPUs in watts
        self.gpu_tdp = 250   # Typical TDP for desktop GPUs in watts
        
    def get_stats(self):
        """Get system stats with power estimation"""
        stats = {
            'timestamp': time.time(),
            'cpu_%': psutil.cpu_percent(interval=0.1),
            'ram_mb': psutil.virtual_memory().used / (1024**2),
            'gpu_mem_mb': 0,
            'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85  # Base CPU power
        }
        
        if self.gpu_available:
            try:
                # TensorFlow GPU memory monitoring
                mem_info = tf.config.experimental.get_memory_info('GPU:0')
                stats.update({
                    'gpu_mem_mb': mem_info['current'] / (1024**2),
                    'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85 + 
                              self.gpu_tdp * 0.5 * 0.75  # Add GPU power estimate
                })
            except:
                pass
                
        return stats

# Initialize monitor
monitor = PowerMonitor()

In [4]:
import time
import torch
import psutil
import os

class PowerMonitor1:
    def __init__(self):
        self.gpu_available = torch.cuda.is_available()
        self.process = psutil.Process(os.getpid())  # Track current process
        
        # Hardware power specifications
        self.cpu_tdp = 65
        self.gpu_tdp = 250
        
    def get_stats(self):
        """Get process-specific stats with power estimation"""
        process_memory = self.process.memory_info()
        
        stats = {
            'timestamp': time.time(),
            'cpu_%': psutil.cpu_percent(interval=0.1),
            'process_ram_mb': process_memory.rss / (1024**2),  # Only this process's RAM
            'gpu_mem_mb': 0,
            'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85
        }
        
        if self.gpu_available:
            try:
                gpu_memory_allocated = torch.cuda.memory_allocated()
                stats.update({
                    'gpu_mem_mb': gpu_memory_allocated / (1024**2),
                    'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85 + 
                              self.gpu_tdp * 0.5 * 0.75
                })
            except Exception as e:
                print(f"Error retrieving GPU memory: {e}")
                
        return stats

# Initialize monitor1
monitor1 = PowerMonitor1()

# Model

In [5]:
import os
import random
import numpy as np
import torch
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from PIL import Image

from transformers import (
    ViTImageProcessor,
    ViTForImageClassification
)


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "google/vit-base-patch16-224-in21k"

INPUT_SIZE = 160
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4
RANDOM_SEED = 42

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
################################################################
#Load the pretrained processor and model
# Configure the processor to use the common 160 × 160 input size
processor = ViTImageProcessor.from_pretrained(
    MODEL_NAME,
    size={
        "height": INPUT_SIZE,
        "width": INPUT_SIZE
    }
)

model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,

    # Binary classification:
    # 0 = real, 1 = fake
    num_labels=2,

    id2label={
        0: "real",
        1: "fake"
    },

    label2id={
        "real": 0,
        "fake": 1
    },

    # The checkpoint does not contain your binary classifier
    ignore_mismatched_sizes=True
)

# All transformer layers remain trainable
model.to(device)

print(model)
print("\nNumber of output classes:", model.config.num_labels)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Trainable parameters:", trainable_parameters)
print("Total parameters:", total_parameters)
##############################################
#dataset class
class DeepfakeViTDataset(Dataset):
    """
    Dataset for OpenCV-loaded deepfake images.

    Images:
        NumPy arrays in BGR format, shape (160, 160, 3)

    Labels:
        0 = real
        1 = fake
    """

    def __init__(
        self,
        images,
        labels,
        processor
    ):
        self.images = images
        self.labels = np.asarray(
            labels,
            dtype=np.int64
        )
        self.processor = processor

        if len(self.images) != len(self.labels):
            raise ValueError(
                "The numbers of images and labels do not match."
            )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = self.images[index]
        label = int(self.labels[index])

        if isinstance(image, np.ndarray):
            if image.ndim != 3 or image.shape[-1] != 3:
                raise ValueError(
                    f"Invalid image shape at index {index}: "
                    f"{image.shape}"
                )

            # OpenCV BGR -> RGB
            image_rgb = image[..., ::-1]

            # Ensure contiguous memory before converting to PIL
            image_rgb = np.ascontiguousarray(
                image_rgb,
                dtype=np.uint8
            )

            image = Image.fromarray(image_rgb)

        elif isinstance(image, str):
            image = Image.open(image).convert("RGB")

        elif isinstance(image, Image.Image):
            image = image.convert("RGB")

        else:
            raise TypeError(
                f"Unsupported image type at index {index}: "
                f"{type(image)}"
            )

        processed = self.processor(
            images=image,
            return_tensors="pt"
        )

        pixel_values = processed[
            "pixel_values"
        ].squeeze(0)

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(
                label,
                dtype=torch.long
            )
        }
    #optimizer once
optimizer = optim.Adam(model.parameters(),lr=LEARNING_RATE)
    #Validation function

def evaluate_vit(
    model,
    data_loader,
    device
):
    model.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for batch in data_loader:
            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            labels = batch["labels"].to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            outputs = model(
                pixel_values=pixel_values,
                labels=labels,

                # Required because the pretrained checkpoint
                # was trained at 224 × 224 but this benchmark
                # uses 160 × 160
                interpolate_pos_encoding=True
            )

            loss = outputs.loss
            logits = outputs.logits

            predictions = torch.argmax(
                logits,
                dim=1
            )

            batch_size = labels.size(0)

            total_loss += (
                loss.item() * batch_size
            )

            correct_predictions += (
                predictions == labels
            ).sum().item()

            total_samples += batch_size

    average_loss = (
        total_loss / total_samples
    )

    accuracy = (
        correct_predictions / total_samples
    )

    return average_loss, accuracy
#training function
def train_vit(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    epochs=10
):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(epochs):

        # Important: evaluate_vit() changes the model to eval mode.
        # Therefore, restore training mode at every epoch.
        model.train()

        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        for batch in train_loader:
            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            labels = batch["labels"].to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            optimizer.zero_grad()

            outputs = model(
                pixel_values=pixel_values,
                labels=labels,
                interpolate_pos_encoding=True
            )

            loss = outputs.loss
            logits = outputs.logits

            loss.backward()
            optimizer.step()

            predictions = torch.argmax(
                logits,
                dim=1
            )

            current_batch_size = labels.size(0)

            running_loss += (
                loss.item()
                * current_batch_size
            )

            correct_predictions += (
                predictions == labels
            ).sum().item()

            total_samples += current_batch_size

        train_loss = (
            running_loss / total_samples
        )

        train_accuracy = (
            correct_predictions / total_samples
        )

        val_loss, val_accuracy = evaluate_vit(
            model,
            val_loader,
            device
        )

        history["train_loss"].append(
            train_loss
        )

        history["train_accuracy"].append(
            train_accuracy
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_accuracy"].append(
            val_accuracy
        )

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Train accuracy: {train_accuracy:.4f} | "
            f"Validation loss: {val_loss:.4f} | "
            f"Validation accuracy: {val_accuracy:.4f}"
        )

    return history

Device: cuda


c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0): ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_

In [6]:
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    roc_curve,
    precision_recall_curve,
    auc
)


def evaluate_vit_complete(
    model,
    test_loader,
    device
):
    model.eval()

    all_labels = []
    all_predictions = []
    all_fake_probabilities = []

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch["pixel_values"].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            labels = batch["labels"].to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            outputs = model(
                pixel_values=pixel_values,
                labels=labels,
                interpolate_pos_encoding=True
            )

            logits = outputs.logits
            loss = outputs.loss

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            # Probability of class 1: fake
            fake_probabilities = probabilities[:, 1]

            predictions = torch.argmax(
                logits,
                dim=1
            )

            current_batch_size = labels.size(0)

            total_loss += (
                loss.item() * current_batch_size
            )

            total_samples += current_batch_size

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_fake_probabilities.extend(
                fake_probabilities.cpu().numpy()
            )

    y_true = np.asarray(all_labels)
    y_pred = np.asarray(all_predictions)
    y_prob = np.asarray(all_fake_probabilities)

    test_loss = total_loss / total_samples

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    balanced_accuracy = balanced_accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )

    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )

    average_precision = average_precision_score(
        y_true,
        y_prob
    )

    pr_precision, pr_recall, _ = precision_recall_curve(
        y_true,
        y_prob
    )

    pr_auc = auc(
        pr_recall,
        pr_precision
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    false_positive_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0.0
    )

    false_negative_rate = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else 0.0
    )

    # Equal Error Rate
    fpr, tpr, thresholds = roc_curve(
        y_true,
        y_prob
    )

    fnr = 1.0 - tpr

    eer_index = np.nanargmin(
        np.abs(fpr - fnr)
    )

    eer = (
        fpr[eer_index]
        + fnr[eer_index]
    ) / 2.0

    eer_threshold = thresholds[eer_index]

    results = {
        "test_loss": test_loss,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1_score": f1,
        "mcc": mcc,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "average_precision": average_precision,
        "eer": eer,
        "eer_threshold": eer_threshold,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate
    }

    print("\n" + "=" * 60)
    print("VIT-16 TEST RESULTS")
    print("=" * 60)

    for metric_name, metric_value in results.items():
        print(
            f"{metric_name:25s}: "
            f"{metric_value:.6f}"
        )

    print("\nConfusion matrix:")
    print(cm)

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                "Real",
                "Fake"
            ],
            digits=4,
            zero_division=0
        )
    )

    return {
        "metrics": results,
        "confusion_matrix": cm,
        "true_labels": y_true,
        "predictions": y_pred,
        "fake_probabilities": y_prob
    }

# Wild deepfake

In [20]:
import h5py
import numpy as np

H5_PATH = (
    r"D:\thesis\dataset\WildDeepfake\leakage_free_subset"
    r"\wilddeepfake_sequence_disjoint_face_preprocessed.h5"
)

with h5py.File(H5_PATH, "r") as h5f:
    # Load image arrays
    train_images = h5f["train_images"][:]
    train_labels = h5f["train_labels"][:]

    val_images = h5f["val_images"][:]
    val_labels = h5f["val_labels"][:]

    test_images = h5f["test_images"][:]
    test_labels = h5f["test_labels"][:]

# Verify dataset sizes
print(f"Total train: {len(train_images)} images")
print(f"Total validation: {len(val_images)} images")
print(f"Total test: {len(test_images)} images")

print(f"Train labels: {len(train_labels)}")
print(f"Validation labels: {len(val_labels)}")
print(f"Test labels: {len(test_labels)}")

# Verify shapes and data types
print("\nArray information:")
print(f"Train images: {train_images.shape}, dtype={train_images.dtype}")
print(f"Validation images: {val_images.shape}, dtype={val_images.dtype}")
print(f"Test images: {test_images.shape}, dtype={test_images.dtype}")

print(f"Train labels: {train_labels.shape}, dtype={train_labels.dtype}")
print(f"Validation labels: {val_labels.shape}, dtype={val_labels.dtype}")
print(f"Test labels: {test_labels.shape}, dtype={test_labels.dtype}")

# Verify class distributions
print("\nClass distribution:")
print(
    f"Train: Real={np.sum(train_labels == 0)}, "
    f"Fake={np.sum(train_labels == 1)}"
)
print(
    f"Validation: Real={np.sum(val_labels == 0)}, "
    f"Fake={np.sum(val_labels == 1)}"
)
print(
    f"Test: Real={np.sum(test_labels == 0)}, "
    f"Fake={np.sum(test_labels == 1)}"
)

Total train: 36000 images
Total validation: 6000 images
Total test: 18000 images
Train labels: 36000
Validation labels: 6000
Test labels: 18000

Array information:
Train images: (36000, 160, 160, 3), dtype=uint8
Validation images: (6000, 160, 160, 3), dtype=uint8
Test images: (18000, 160, 160, 3), dtype=uint8
Train labels: (36000,), dtype=uint8
Validation labels: (6000,), dtype=uint8
Test labels: (18000,), dtype=uint8

Class distribution:
Train: Real=9000, Fake=27000
Validation: Real=1500, Fake=4500
Test: Real=4500, Fake=13500


In [8]:
#creating dataloaders
train_dataset = DeepfakeViTDataset(
    train_images,
    train_labels,
    processor
)

val_dataset = DeepfakeViTDataset(
    val_images,
    val_labels,
    processor
)

test_dataset = DeepfakeViTDataset(
    test_images,
    test_labels,
    processor
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,          # Safest setting for Windows/Jupyter
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 2250
Validation batches: 375
Testing batches: 1125


In [12]:
history = train_vit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS
)

Epoch 01/10 | Train loss: 0.2181 | Train accuracy: 0.9124 | Validation loss: 0.2012 | Validation accuracy: 0.9225
Epoch 02/10 | Train loss: 0.0717 | Train accuracy: 0.9744 | Validation loss: 0.1797 | Validation accuracy: 0.9377
Epoch 03/10 | Train loss: 0.0451 | Train accuracy: 0.9842 | Validation loss: 0.2345 | Validation accuracy: 0.9225
Epoch 04/10 | Train loss: 0.0351 | Train accuracy: 0.9885 | Validation loss: 0.2627 | Validation accuracy: 0.9308
Epoch 05/10 | Train loss: 0.0287 | Train accuracy: 0.9899 | Validation loss: 0.2273 | Validation accuracy: 0.9303
Epoch 06/10 | Train loss: 0.0271 | Train accuracy: 0.9911 | Validation loss: 0.1982 | Validation accuracy: 0.9433
Epoch 07/10 | Train loss: 0.0256 | Train accuracy: 0.9915 | Validation loss: 0.2222 | Validation accuracy: 0.9357
Epoch 08/10 | Train loss: 0.0230 | Train accuracy: 0.9922 | Validation loss: 0.2792 | Validation accuracy: 0.9257
Epoch 09/10 | Train loss: 0.0215 | Train accuracy: 0.9936 | Validation loss: 0.3031 | Va

In [13]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [14]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [ ]:
test_results = evaluate_vit_complete(
    model=model,
    test_loader=test_loader,
    device=device
)


VIT-16 TEST RESULTS
test_loss                     : 0.931596
accuracy                      : 0.805223
balanced_accuracy             : 0.740334
precision                     : 0.870123
recall                        : 0.870144
specificity                   : 0.610432
f1_score                      : 0.870145
mcc                           : 0.502254
roc_auc                       : 0.740323
pr_auc                        : 0.870112
average_precision             : 0.870123
eer                           : 0.194823
eer_threshold                 : 0.500029
false_positive_rate           : 0.389633
false_negative_rate           : 0.129957

Confusion Matrix:
[[ 2747  1753]
 [ 1753 11747]]

Classification Report:
              precision    recall  f1-score   support

        Real     0.6104    0.6104    0.6104      4500
        Fake     0.8701    0.8701    0.8701     13500

    accuracy                         0.8052     18000
   macro avg     0.7403    0.7403    0.7403     18000
weighted avg     0

In [ ]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time
0.805223	0.870123	0.870144	0.870145	0.740323	0.870112	0.194823

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
#print(f"RAM Used: {end['ram_mb'] - start['ram_mb']:.1f} MB")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 14.5%
Time Usage: 348.0 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [18]:
end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 27.8%
Time Usage: 353.0 s
GPU Memory Used: 1326.1 MB
Power Consumption: 112W


save the model

In [19]:
SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_patch16_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "input_size": INPUT_SIZE,
        "label_mapping": {
            0: "real",
            1: "fake"
        }
    },
    os.path.join(
        SAVE_DIR,
        "training_checkpoint.pt"
    )
)

print("ViT-16 model saved:", SAVE_DIR)

ViT-16 model saved: D:\thesis\results\vit_base_patch16_160


In [11]:
# ============================================================TRAINING AND VALIDATION CURVES
# TRAINING AND VALIDATION CURVES
# Run after model.fit(...)
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# Available values recorded during training
history_data = history.history

print("Available history metrics:")
print(list(history_data.keys()))

epochs = np.arange(1, len(history_data["train_loss"]) + 1)


# ------------------------------------------------------------
# 1. Training and validation loss
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    history_data["train_loss"],
    label="Training Loss"
)

plt.plot(
    epochs,
    history_data["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("ViT-16 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 2. Training and validation accuracy
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    history["train_accuracy"],
    label="Training Accuracy"
)

plt.plot(
    epochs,
    history_data["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ViT-16 Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()







NameError: name 'history' is not defined

load the model

In [10]:
import os
import torch

from transformers import (
    ViTImageProcessor,
    ViTForImageClassification
)


SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_patch16_160"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Model directory exists:", os.path.isdir(SAVE_DIR))


# Load the saved processor
processor = ViTImageProcessor.from_pretrained(
    SAVE_DIR
)

# Load the saved model configuration and final trained weights
model = ViTForImageClassification.from_pretrained(
    SAVE_DIR
)

# Move the model to GPU or CPU
model = model.to(device)

# Inference mode
model.eval()


print("ViT-16 model loaded successfully")
print("Number of classes:", model.config.num_labels)
print("Label mapping:", model.config.id2label)

print(
    "Total parameters:",
    sum(parameter.numel() for parameter in model.parameters())
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)

Device: cuda
Model directory exists: True
ViT-16 model loaded successfully
Number of classes: 2
Label mapping: {0: 'real', 1: 'fake'}
Total parameters: 85800194
Trainable parameters: 85800194


In [12]:
# ============================================================
# PYTORCH ViT-16 WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# Make sure the loaded model is on the correct device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(test_loader.dataset)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ------------------------------------------------------------
# GPU/model warm-up
# ------------------------------------------------------------

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():
    for batch_index, batch in enumerate(test_loader):

        if batch_index >= WARMUP_BATCHES:
            break

        pixel_values = batch["pixel_values"].to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        outputs = model(
            pixel_values=pixel_values,
            interpolate_pos_encoding=True
        )

        _ = outputs.logits


# Wait until all asynchronous GPU work is complete
if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ------------------------------------------------------------
# Five repeated inference runs
# ------------------------------------------------------------

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):
    print(
        f"\nStarting ViT-16 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous GPU operations remain in the queue
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():
        for batch in test_loader:

            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            outputs = model(
                pixel_values=pixel_values,
                interpolate_pos_encoding=True
            )

            last_logits = outputs.logits

            processed_images += pixel_values.size(0)

    # PyTorch CUDA operations are asynchronous.
    # Synchronize before stopping the timer.
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    # Access one output after synchronization to verify completion
    if last_logits is not None:
        last_output_value = float(
            last_logits[-1, 0]
            .detach()
            .cpu()
            .item()
        )
    else:
        raise RuntimeError(
            "No images were processed during inference."
        )

    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number
    run_summary["processed_images"] = processed_images
    run_summary["last_output_value"] = last_output_value

    run_results.append(run_summary)

    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )

    del last_logits
    del outputs


results_df = pd.DataFrame(run_results)

print("\nIndividual ViT-16 profiling runs:")
display(results_df)

Device: cuda
Test images: 18000
Test batches: 1125

Performing warm-up using 3 batches...
Warm-up completed.

Starting ViT-16 resource run 1/5
Run 1: 135.11 seconds | 7.5060 ms/image | 133.23 images/s

Starting ViT-16 resource run 2/5
Run 2: 138.31 seconds | 7.6837 ms/image | 130.15 images/s

Starting ViT-16 resource run 3/5
Run 3: 128.08 seconds | 7.1154 ms/image | 140.54 images/s

Starting ViT-16 resource run 4/5
Run 4: 122.29 seconds | 6.7938 ms/image | 147.19 images/s

Starting ViT-16 resource run 5/5
Run 5: 124.43 seconds | 6.9130 ms/image | 144.65 images/s

Individual ViT-16 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,135.107199,7.505955,133.227542,2.928078,9.856250,8785.406250,8795.094853,8800.636719,9.688603,15.230469,...,99.699115,100.0,50.769027,100,50.490930,132.057,1.901851,1,18000,-4.228389
1,138.306761,7.683709,130.145481,2.951623,7.421875,8796.542969,8796.675555,8801.316406,0.132586,4.773438,...,99.740260,100.0,48.838095,100,51.555777,132.282,1.987911,2,18000,-4.228389
2,128.078097,7.115450,140.539252,2.895689,6.659375,8796.535156,8796.701059,8801.382812,0.165902,4.847656,...,99.682243,100.0,52.201869,100,54.834716,131.741,1.954138,3,18000,-4.228389
3,122.288362,6.793798,147.193075,2.926535,6.640625,8796.578125,8796.801041,8801.457031,0.222916,4.878906,...,99.706458,100.0,54.810176,100,56.787069,129.449,1.931451,4,18000,-4.228389
4,124.434296,6.913016,144.654654,2.940664,6.718750,8796.671875,8796.817140,8801.468750,0.145265,4.796875,...,99.728945,100.0,51.247822,100,57.291187,133.665,1.982862,5,18000,-4.228389


In [13]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================

metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("ViT-16 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


ViT-16 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,129.642943,6.866375,121.117212,138.168674
1,latency_ms_per_image,7.202386,0.381465,6.728734,7.676037
2,throughput_images_per_s,139.152001,7.298615,130.089573,148.214429
3,average_cpu_percent,2.928518,0.020996,2.902449,2.954587
4,peak_cpu_percent,7.459375,1.378885,5.747263,9.171487
5,average_ram_mb,8796.417929,0.742160,8795.496416,8797.339443
6,peak_ram_mb,8801.252344,0.349589,8800.818272,8801.686415
7,average_incremental_ram_mb,2.071054,4.258479,-3.216546,7.358655
8,peak_incremental_ram_mb,6.905469,4.654001,1.126763,12.684174
9,average_gpu_memory_mb,2636.445779,0.023271,2636.416884,2636.474674



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 7.202 ± 0.381 (95% CI: 6.729–7.676)
peak_ram_mb: 8801.252 ± 0.350 (95% CI: 8800.818–8801.686)
peak_gpu_memory_mb: 2636.734 ± 0.000 (95% CI: 2636.734–2636.734)
average_gpu_utilization_percent: 51.573 ± 2.186 (95% CI: 48.860–54.287)
average_gpu_power_w: 54.192 ± 3.058 (95% CI: 50.395–57.989)


genralization

In [15]:
print("\nTest results of wild deepfake dataset on Celeb-DF(V2) (Vit-16):")
test_dataset = DeepfakeViTDataset(test_celeb,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)




Test results of wild deepfake dataset on Celeb-DF(V2) (Vit-16):

VIT-16 TEST RESULTS
test_loss                : 0.721428
accuracy                 : 0.748868
balanced_accuracy        : 0.593383
precision                : 0.925279
recall                   : 0.785714
specificity              : 0.401051
f1_score                 : 0.849804
mcc                      : 0.130182
roc_auc                  : 0.644029
pr_auc                   : 0.940404
average_precision        : 0.940416
eer                      : 0.398774
eer_threshold            : 0.900920
false_positive_rate      : 0.598949
false_negative_rate      : 0.214286

Confusion matrix:
[[ 229  342]
 [1155 4235]]

Classification report:
              precision    recall  f1-score   support

        Real     0.1655    0.4011    0.2343       571
        Fake     0.9253    0.7857    0.8498      5390

    accuracy                         0.7489      5961
   macro avg     0.5454    0.5934    0.5420      5961
weighted avg     0.8525    0.748

In [17]:
#dfc on wilddeepfake
print("\nTest results of wild deepfake dataset on DFC (Vit-16):")
test_dataset = DeepfakeViTDataset(test_hog,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)



Test results of wild deepfake dataset on DFC (Vit-16):

VIT-16 TEST RESULTS
test_loss                : 1.367764
accuracy                 : 0.544000
balanced_accuracy        : 0.544000
precision                : 0.538687
recall                   : 0.612667
specificity              : 0.475333
f1_score                 : 0.573300
mcc                      : 0.088842
roc_auc                  : 0.575246
pr_auc                   : 0.610247
average_precision        : 0.610594
eer                      : 0.450667
eer_threshold            : 0.618950
false_positive_rate      : 0.524667
false_negative_rate      : 0.387333

Confusion matrix:
[[713 787]
 [581 919]]

Classification report:
              precision    recall  f1-score   support

        Real     0.5510    0.4753    0.5104      1500
        Fake     0.5387    0.6127    0.5733      1500

    accuracy                         0.5440      3000
   macro avg     0.5448    0.5440    0.5418      3000
weighted avg     0.5448    0.5440    0.5418  

In [19]:
print("\nTest results of wild deepfake dataset on FF++ (Vit-16):")
#ff++ on wilddeepfake
test_dataset = DeepfakeViTDataset(test_ff,test_ff_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)


Test results of wild deepfake dataset on FF++ (Vit-16):

VIT-16 TEST RESULTS
test_loss                : 1.878041
accuracy                 : 0.510707
balanced_accuracy        : 0.530745
precision                : 0.440867
recall                   : 0.650824
specificity              : 0.410665
f1_score                 : 0.525656
mcc                      : 0.062296
roc_auc                  : 0.547683
pr_auc                   : 0.461691
average_precision        : 0.462280
eer                      : 0.468796
eer_threshold            : 0.764328
false_positive_rate      : 0.589335
false_negative_rate      : 0.349176

Confusion matrix:
[[593 851]
 [360 671]]

Classification report:
              precision    recall  f1-score   support

        Real     0.6222    0.4107    0.4948      1444
        Fake     0.4409    0.6508    0.5257      1031

    accuracy                         0.5107      2475
   macro avg     0.5316    0.5307    0.5102      2475
weighted avg     0.5467    0.5107    0.5076 

# Celeb

In [22]:
import os
import cv2
import numpy as np

SAVE_ROOT = r'D:\thesis\celeb_processed'

def load_split(split_name, class_name):
    """Reload saved frames, grouped by video."""
    base = os.path.join(SAVE_ROOT, split_name, class_name)
    nested, ids = [], []
    for vid_id in sorted(os.listdir(base)):
        vid_dir = os.path.join(base, vid_id)
        frames = [cv2.imread(os.path.join(vid_dir, f))
                  for f in sorted(os.listdir(vid_dir))]
        if frames:
            nested.append(frames)
            ids.append(vid_id)
    return nested, ids

# Reload ALL six splits
print("Loading frames...")
real_train_final,  real_train_ids  = load_split('train', 'real')
synth_train_final, synth_train_ids = load_split('train', 'fake')
real_val_final,    real_val_ids    = load_split('val',   'real')
synth_val_final,   synth_val_ids   = load_split('val',   'fake')
real_test_final,   real_test_ids   = load_split('test',  'real')
synth_test_final,  synth_test_ids  = load_split('test',  'fake')

print("✅ All frames reloaded")
print("Train -> real videos:", len(real_train_final), " fake videos:", len(synth_train_final))
print("Val   -> real videos:", len(real_val_final),   " fake videos:", len(synth_val_final))
print("Test  -> real videos:", len(real_test_final),  " fake videos:", len(synth_test_final))
print("Example frame shape:", np.shape(real_train_final[0][0]))  # expect (160,160,3)
import numpy as np


def combine_split(real_videos, fake_videos):
    """
    Flatten video-grouped frames into one image array and create labels.

    real_videos: list of videos, where each video is a list of frames
    fake_videos: list of videos, where each video is a list of frames

    Returns
    -------
    images : NumPy array with shape (N, 160, 160, 3)
    labels : NumPy array with shape (N,)
             0 = real, 1 = fake
    """

    # Flatten frames from all real videos
    real_frames = [
        frame
        for video_frames in real_videos
        for frame in video_frames
        if frame is not None
    ]

    # Flatten frames from all fake videos
    fake_frames = [
        frame
        for video_frames in fake_videos
        for frame in video_frames
        if frame is not None
    ]

    if len(real_frames) == 0:
        raise ValueError("No real frames were found.")

    if len(fake_frames) == 0:
        raise ValueError("No fake frames were found.")

    # Convert to NumPy arrays
    real_frames = np.stack(real_frames).astype(np.uint8)
    fake_frames = np.stack(fake_frames).astype(np.uint8)

    # Combine images
    images = np.concatenate(
        [real_frames, fake_frames],
        axis=0
    )

    # Create labels
    real_labels = np.zeros(
        len(real_frames),
        dtype=np.uint8
    )

    fake_labels = np.ones(
        len(fake_frames),
        dtype=np.uint8
    )

    labels = np.concatenate(
        [real_labels, fake_labels],
        axis=0
    )

    return images, labels
# Training set
train_celeb, train_labels = combine_split(
    real_train_final,
    synth_train_final
)

# Validation set
val_celeb, val_labels = combine_split(
    real_val_final,
    synth_val_final
)

# Testing set
test_celeb, test_labels = combine_split(
    real_test_final,
    synth_test_final
)
print("\nTRAIN")
print("Images:", train_celeb.shape)
print("Labels:", train_labels.shape)
print("Real:", np.sum(train_labels == 0))
print("Fake:", np.sum(train_labels == 1))

print("\nVALIDATION")
print("Images:", val_celeb.shape)
print("Labels:", val_labels.shape)
print("Real:", np.sum(val_labels == 0))
print("Fake:", np.sum(val_labels == 1))

print("\nTEST")
print("Images:", test_celeb.shape)
print("Labels:", test_labels.shape)
print("Real:", np.sum(test_labels == 0))
print("Fake:", np.sum(test_labels == 1))

print("\nData types")
print("Train images:", train_celeb.dtype)
print("Train labels:", train_labels.dtype)

Loading frames...
✅ All frames reloaded
Train -> real videos: 354  fake videos: 3383
Val   -> real videos: 59  fake videos: 563
Test  -> real videos: 177  fake videos: 1693
Example frame shape: (160, 160, 3)

TRAIN
Images: (11899, 160, 160, 3)
Labels: (11899,)
Real: 1142
Fake: 10757

VALIDATION
Images: (1969, 160, 160, 3)
Labels: (1969,)
Real: 182
Fake: 1787

TEST
Images: (5961, 160, 160, 3)
Labels: (5961,)
Real: 571
Fake: 5390

Data types
Train images: uint8
Train labels: uint8


In [24]:
#creating dataloaders
train_dataset = DeepfakeViTDataset(
    train_celeb,
    train_labels,
    processor
)

val_dataset = DeepfakeViTDataset(
    val_celeb,
    val_labels,
    processor
)

test_dataset = DeepfakeViTDataset(
    test_celeb,
    test_labels,
    processor
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,          # Safest setting for Windows/Jupyter
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 744
Validation batches: 124
Testing batches: 373


In [9]:
history = train_vit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS
)

Epoch 01/10 | Train loss: 0.2760 | Train accuracy: 0.9054 | Validation loss: 0.2719 | Validation accuracy: 0.8928
Epoch 02/10 | Train loss: 0.1677 | Train accuracy: 0.9384 | Validation loss: 0.1906 | Validation accuracy: 0.9284
Epoch 03/10 | Train loss: 0.0923 | Train accuracy: 0.9673 | Validation loss: 0.1593 | Validation accuracy: 0.9426
Epoch 04/10 | Train loss: 0.0606 | Train accuracy: 0.9791 | Validation loss: 0.1816 | Validation accuracy: 0.9431
Epoch 05/10 | Train loss: 0.0432 | Train accuracy: 0.9854 | Validation loss: 0.2265 | Validation accuracy: 0.9416
Epoch 06/10 | Train loss: 0.0431 | Train accuracy: 0.9855 | Validation loss: 0.2505 | Validation accuracy: 0.9426
Epoch 07/10 | Train loss: 0.0392 | Train accuracy: 0.9868 | Validation loss: 0.2289 | Validation accuracy: 0.9121
Epoch 08/10 | Train loss: 0.0336 | Train accuracy: 0.9891 | Validation loss: 0.1842 | Validation accuracy: 0.9360
Epoch 09/10 | Train loss: 0.0398 | Train accuracy: 0.9871 | Validation loss: 0.2103 | Va

In [10]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [11]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [ ]:
test_results = evaluate_vit_complete(
    model=model,
    test_loader=test_loader,
    device=device
)


VIT-16 TEST RESULTS
test_loss                : 0.185802
accuracy                 : 0.948834
balanced_accuracy        : 0.771286
precision                : 0.954261
recall                   : 0.990909
specificity              : 0.551664
f1_score                 : 0.972240
mcc                      : 0.666871
roc_auc                  : 0.950191
pr_auc                   : 0.993810
average_precision        : 0.993810
eer                      : 0.120439
eer_threshold            : 0.989590
false_positive_rate      : 0.448336
false_negative_rate      : 0.009091

Confusion matrix:
[[315 256]
 [ 49 5341]]

Classification report:
              precision    recall  f1-score   support

        Real   0.865385  0.551664  0.673797       571
        Fake   0.954261  0.990909  0.972240      5390

    accuracy                       0.948834      5961
   macro avg   0.909823  0.771286  0.823018      5961
weighted avg   0.945748  0.948834  0.943652      5961



In [13]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 7.0%
Time Usage: 62.7 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [14]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 10.0%
Time Usage: 64.8 s
GPU Memory Used: 1325.1 MB
Power Consumption: 93W


save the model

In [15]:
SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_celeb_patch16_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "input_size": INPUT_SIZE,
        "label_mapping": {
            0: "real",
            1: "fake"
        }
    },
    os.path.join(
        SAVE_DIR,
        "training_checkpoint.pt"
    )
)

print("ViT-16 celeb model saved:", SAVE_DIR)

ViT-16 celeb model saved: D:\thesis\results\vit_base_celeb_patch16_160


In [16]:
# ============================================================TRAINING AND VALIDATION CURVES
# TRAINING AND VALIDATION CURVES
# Run after model.fit(...)
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# Available values recorded during training
history_data = history.history

print("Available history metrics:")
print(list(history_data.keys()))

epochs = np.arange(1, len(history_data["train_loss"]) + 1)


# ------------------------------------------------------------
# 1. Training and validation loss
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    history_data["train_loss"],
    label="Training Loss"
)

plt.plot(
    epochs,
    history_data["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("ViT-16 Celeb-DF(V2) Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 2. Training and validation accuracy
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    history["train_accuracy"],
    label="Training Accuracy"
)

plt.plot(
    epochs,
    history_data["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ViT-16 Celeb-DF(V2) Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()







AttributeError: 'dict' object has no attribute 'history'

load the model

In [25]:
import os
import torch

from transformers import (
    ViTImageProcessor,
    ViTForImageClassification
)


SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_celeb_patch16_160"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Model directory exists:", os.path.isdir(SAVE_DIR))


# Load the saved processor
processor = ViTImageProcessor.from_pretrained(
    SAVE_DIR
)

# Load the saved model configuration and final trained weights
model = ViTForImageClassification.from_pretrained(
    SAVE_DIR
)

# Move the model to GPU or CPU
model = model.to(device)

# Inference mode
model.eval()


print("ViT-16 model loaded successfully")
print("Number of classes:", model.config.num_labels)
print("Label mapping:", model.config.id2label)

print(
    "Total parameters:",
    sum(parameter.numel() for parameter in model.parameters())
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)

Device: cuda
Model directory exists: True
ViT-16 model loaded successfully
Number of classes: 2
Label mapping: {0: 'real', 1: 'fake'}
Total parameters: 85800194
Trainable parameters: 85800194


In [31]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_5088\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [32]:
# ============================================================
# PYTORCH ViT-16 WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# Make sure the loaded model is on the correct device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(test_loader.dataset)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ------------------------------------------------------------
# GPU/model warm-up
# ------------------------------------------------------------

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():
    for batch_index, batch in enumerate(test_loader):

        if batch_index >= WARMUP_BATCHES:
            break

        pixel_values = batch["pixel_values"].to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        outputs = model(
            pixel_values=pixel_values,
            interpolate_pos_encoding=True
        )

        _ = outputs.logits


# Wait until all asynchronous GPU work is complete
if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ------------------------------------------------------------
# Five repeated inference runs
# ------------------------------------------------------------

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):
    print(
        f"\nStarting ViT-16 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous GPU operations remain in the queue
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():
        for batch in test_loader:

            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            outputs = model(
                pixel_values=pixel_values,
                interpolate_pos_encoding=True
            )

            last_logits = outputs.logits

            processed_images += pixel_values.size(0)

    # PyTorch CUDA operations are asynchronous.
    # Synchronize before stopping the timer.
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    # Access one output after synchronization to verify completion
    if last_logits is not None:
        last_output_value = float(
            last_logits[-1, 0]
            .detach()
            .cpu()
            .item()
        )
    else:
        raise RuntimeError(
            "No images were processed during inference."
        )

    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number
    run_summary["processed_images"] = processed_images
    run_summary["last_output_value"] = last_output_value

    run_results.append(run_summary)

    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )

    del last_logits
    del outputs


results_df = pd.DataFrame(run_results)

print("\nIndividual ViT-16 profiling runs:")
display(results_df)

Device: cuda
Test images: 5961
Test batches: 373

Performing warm-up using 3 batches...
Warm-up completed.

Starting ViT-16 resource run 1/5
Run 1: 100.38 seconds | 16.8390 ms/image | 59.39 images/s

Starting ViT-16 resource run 2/5
Run 2: 103.78 seconds | 17.4091 ms/image | 57.44 images/s

Starting ViT-16 resource run 3/5
Run 3: 87.16 seconds | 14.6212 ms/image | 68.39 images/s

Starting ViT-16 resource run 4/5
Run 4: 157.52 seconds | 26.4248 ms/image | 37.84 images/s

Starting ViT-16 resource run 5/5
Run 5: 72.61 seconds | 12.1807 ms/image | 82.10 images/s

Individual ViT-16 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,100.377287,16.839001,59.385945,2.571750,6.234375,348.585938,663.230021,894.285156,314.644083,545.699219,...,99.526185,100.0,29.234414,82,24.811589,77.210,0.659783,1,5961,-3.793367
1,103.775881,17.409140,57.441093,2.316270,5.328125,894.957031,899.756487,904.726562,4.799456,9.769531,...,99.571590,100.0,29.086809,100,22.838584,56.712,0.641469,2,5961,-3.793367
2,87.156875,14.621183,68.393916,2.438386,4.928125,900.195312,901.091861,905.785156,0.896548,5.589844,...,99.427793,100.0,31.506812,100,27.705001,96.848,0.649314,3,5961,-3.793367
3,157.518039,26.424767,37.843285,2.783205,6.271875,901.125000,901.443804,906.074219,0.318804,4.949219,...,99.666667,100.0,19.910417,100,17.948257,53.421,0.785700,4,5961,-3.793367
4,72.609154,12.180700,82.097087,2.665928,4.928125,901.859375,906.898091,911.605469,5.038716,9.746094,...,99.376026,100.0,33.628900,100,32.869923,94.207,0.647175,5,5961,-3.793367


In [35]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("ViT-16 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


ViT-16 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,104.287447,32.194789,64.312337,144.262557
1,latency_ms_per_image,17.494958,5.400904,10.788850,24.201066
2,throughput_images_per_s,61.032265,16.211542,40.902978,81.161553
3,average_cpu_percent,2.555108,0.183885,2.326785,2.783431
4,peak_cpu_percent,5.538125,0.672951,4.702546,6.373704
5,average_ram_mb,854.484053,106.949095,721.689212,987.278893
6,peak_ram_mb,904.495312,6.304993,896.666630,912.323995
7,average_incremental_ram_mb,65.139521,139.494123,-108.065309,238.344352
8,peak_incremental_ram_mb,115.150781,240.694462,-183.710867,414.012429
9,average_gpu_memory_mb,2556.626933,0.115369,2556.483684,2556.770183



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 17.495 ± 5.401 (95% CI: 10.789–24.201)
peak_ram_mb: 904.495 ± 6.305 (95% CI: 896.667–912.324)
peak_gpu_memory_mb: 2557.113 ± 0.000 (95% CI: 2557.113–2557.113)
average_gpu_utilization_percent: 28.673 ± 5.241 (95% CI: 22.166–35.181)
average_gpu_power_w: 25.235 ± 5.555 (95% CI: 18.338–32.132)


#genralization

In [37]:
#wild deepfake on celeb
print("\nTest results of Celeb-DF(V2) on wild deepfake dataset (ViT-16):")
test_dataset = DeepfakeViTDataset(test_images,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)




Test results of Celeb-DF(V2) on wild deepfake dataset (ViT-16):

VIT-16 TEST RESULTS
test_loss                : 0.911117
accuracy                 : 0.586111
balanced_accuracy        : 0.533481
precision                : 0.770186
recall                   : 0.638741
specificity              : 0.428222
f1_score                 : 0.698332
mcc                      : 0.059799
roc_auc                  : 0.584066
pr_auc                   : 0.824261
average_precision        : 0.824273
eer                      : 0.456185
eer_threshold            : 0.665180
false_positive_rate      : 0.571778
false_negative_rate      : 0.361259

Confusion matrix:
[[1927 2573]
 [4877 8623]]

Classification report:
              precision    recall  f1-score   support

        Real     0.2832    0.4282    0.3409      4500
        Fake     0.7702    0.6387    0.6983     13500

    accuracy                         0.5861     18000
   macro avg     0.5267    0.5335    0.5196     18000
weighted avg     0.6484    0.586

In [39]:
#DFC on celeb
print("\nTest results of Celeb-DF(V2) on DFC dataset (ViT-16):")
test_dataset = DeepfakeViTDataset(test_hog,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)



Test results of Celeb-DF(V2) on DFC dataset (ViT-16):

VIT-16 TEST RESULTS
test_loss                : 1.709286
accuracy                 : 0.428667
balanced_accuracy        : 0.428667
precision                : 0.450186
recall                   : 0.644667
specificity              : 0.212667
f1_score                 : 0.530154
mcc                      : -0.158189
roc_auc                  : 0.437942
pr_auc                   : 0.477326
average_precision        : 0.478253
eer                      : 0.542333
eer_threshold            : 0.870763
false_positive_rate      : 0.787333
false_negative_rate      : 0.355333

Confusion matrix:
[[ 319 1181]
 [ 533  967]]

Classification report:
              precision    recall  f1-score   support

        Real     0.3744    0.2127    0.2713      1500
        Fake     0.4502    0.6447    0.5302      1500

    accuracy                         0.4287      3000
   macro avg     0.4123    0.4287    0.4007      3000
weighted avg     0.4123    0.4287    0.40

In [41]:
#FF++ on celeb
print("\nTest results of Celeb-DF(V2) on FF++ dataset (ViT-16):")
test_dataset = DeepfakeViTDataset(test_ff,test_ff_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)


Test results of Celeb-DF(V2) on FF++ dataset (ViT-16):

VIT-16 TEST RESULTS
test_loss                : 1.972616
accuracy                 : 0.545455
balanced_accuracy        : 0.602551
precision                : 0.476983
recall                   : 0.944714
specificity              : 0.260388
f1_score                 : 0.633908
mcc                      : 0.266140
roc_auc                  : 0.783115
pr_auc                   : 0.714919
average_precision        : 0.715114
eer                      : 0.278382
eer_threshold            : 0.996991
false_positive_rate      : 0.739612
false_negative_rate      : 0.055286

Confusion matrix:
[[ 376 1068]
 [  57  974]]

Classification report:
              precision    recall  f1-score   support

        Real     0.8684    0.2604    0.4006      1444
        Fake     0.4770    0.9447    0.6339      1031

    accuracy                         0.5455      2475
   macro avg     0.6727    0.6026    0.5173      2475
weighted avg     0.7053    0.5455    0.49

# DFC

In [24]:
import h5py
import numpy as np
# Open the HDF5 file in read mode
with h5py.File('D://thesis//dataset//deepfake dataset//resized_images.h5', 'r') as h5f:
    # Access each dataset
    celeb = np.array(h5f['celeb'])
    ffhq = np.array(h5f['ffhq'])
    gdwct = np.array(h5f['gdwct'])
    attgan = np.array(h5f['attgan'])
    stargan = np.array(h5f['stargan'])
    stylegan2 = np.array(h5f['stylegan2'])
    stylegan = np.array(h5f['stylegan'])

# Now, 'celeb', 'ffhq', etc., are NumPy arrays containing your datasets
print(f"celeb shape: {celeb.shape}, dtype: {celeb.dtype}")
print(f"ffhq shape: {ffhq.shape}, dtype: {ffhq.dtype}")
print(f"ffhq shape: {gdwct.shape}, dtype: {gdwct.dtype}")
print(f"ffhq shape: {attgan.shape}, dtype: {attgan.dtype}")
print(f"ffhq shape: {stargan.shape}, dtype: {stargan.dtype}")
print(f"ffhq shape: {stylegan2.shape}, dtype: {stylegan2.dtype}")
print(f"ffhq shape: {stylegan.shape}, dtype: {stylegan.dtype}")
# Repeat for other datasets as needed
import cv2
# Function to resize images from (224, 224) to (160, 160)
def resize_images(image_array, target_size=(160, 160)):
    resized_images = np.array([cv2.resize(img, target_size) for img in image_array])
    return resized_images

celeb = resize_images(celeb, target_size=(160, 160))
ffhq = resize_images(ffhq, target_size=(160, 160))
gdwct = resize_images(gdwct, target_size=(160, 160))
attgan = resize_images(attgan, target_size=(160, 160))
stargan = resize_images(stargan, target_size=(160, 160))
stylegan = resize_images(stylegan, target_size=(160, 160))
stylegan2 = resize_images(stylegan2, target_size=(160, 160))
import random
# Randomly select 2500 distinct images
random_indices = random.sample(range(len(celeb)), 2500)  # Get 2500 random indices
celeb = celeb[random_indices]  # Select the random subse

import random
# Randomly select 2500 distinct images
random_indices = random.sample(range(len(ffhq)), 2500)  # Get 2500 random indices
ffhq = ffhq[random_indices]  # Select the random subse
print(f"celeb shape: {celeb.shape}, dtype: {celeb.dtype}")
print(f"ffhq shape: {ffhq.shape}, dtype: {ffhq.dtype}")
print(f"gdwct shape: {gdwct.shape}, dtype: {gdwct.dtype}")
print(f"attagan shape: {attgan.shape}, dtype: {attgan.dtype}")
print(f"stargan shape: {stargan.shape}, dtype: {stargan.dtype}")
print(f"stylegan2 shape: {stylegan2.shape}, dtype: {stylegan2.dtype}")
print(f"stylegan shape: {stylegan.shape}, dtype: {stylegan.dtype}")
import random
import numpy as np

def split_data(data, train_ratio=0.7):
    """
    Splits data into training and testing sets based on the specified ratio.

    Parameters:
        data (list or np.array): The dataset to split.
        train_ratio (float): The ratio of the data to include in the training set.

    Returns:
        tuple: Two datasets - train and test.
    """
    # Shuffle the data
    random.shuffle(data)

    # Calculate the split index
    split_index = int(len(data) * train_ratio)

    # Split the data
    train_data = data[:split_index]
    test_data = data[split_index:]

    return train_data, test_data

# Split `celeb` into 70% train and 30% test
celeb_train_hog, celeb_test_hog = split_data(celeb, train_ratio=0.7)

# Split `ffhq` into 70% train and 30% test
ffhq_train_hog, ffhq_test_hog = split_data(ffhq, train_ratio=0.7)

# Split `attgan` into 70% train and 30% test
attgan_train_hog, attgan_test_hog = split_data(attgan, train_ratio=0.7)

# Split `stargan` into 70% train and 30% test
stargan_train_hog, stargan_test_hog = split_data(stargan, train_ratio=0.7)

# Split `gdwct` into 70% train and 30% test
gdwct_train_hog, gdwct_test_hog = split_data(gdwct, train_ratio=0.7)

# Split `stylegan2` into 70% train and 30% test_hog
stylegan2_train_hog, stylegan2_test_hog = split_data(stylegan2, train_ratio=0.7)

# Split `stylegan` into 70% train and 30% test_hog
stylegan_train_hog, stylegan_test_hog = split_data(stylegan, train_ratio=0.7)

# Convert to NumPy arrays if needed
celeb_train_hog, celeb_test_hog = np.array(celeb_train_hog), np.array(celeb_test_hog)
ffhq_train_hog, ffhq_test_hog = np.array(ffhq_train_hog), np.array(ffhq_test_hog)
attgan_train_hog, attgan_test_hog = np.array(attgan_train_hog), np.array(attgan_test_hog)
stargan_train_hog, stargan_test_hog = np.array(stargan_train_hog), np.array(stargan_test_hog)
gdwct_train_hog, gdwct_test_hog = np.array(gdwct_train_hog), np.array(gdwct_test_hog)
stylegan2_train_hog, stylegan2_test_hog = np.array(stylegan2_train_hog), np.array(stylegan2_test_hog)
stylegan_train_hog, stylegan_test_hog = np.array(stylegan_train_hog), np.array(stylegan_test_hog)

# Print results for verification
print(f"celeb_train: {len(celeb_train_hog)} images, celeb_test: {len(celeb_test_hog)} images")
print(f"ffhq_train: {len(ffhq_train_hog)} images, ffhq_test: {len(ffhq_test_hog)} images")
print(f"attgan_train: {len(attgan_train_hog)} images, attgan_test: {len(attgan_test_hog)} images")
print(f"stargan_train: {len(stargan_train_hog)} images, stargan_test: {len(stargan_test_hog)} images")
print(f"gdwct_train: {len(gdwct_train_hog)} images, gdwct_test: {len(gdwct_test_hog)} images")
print(f"stylegan2_train: {len(stylegan2_train_hog)} images, stylegan2_test: {len(stylegan2_test_hog)} images")
print(f"stylegan_train: {len(stylegan_train_hog)} images, stylegan_test: {len(stylegan_test_hog)} images")

########################################################################################################################################
#######################################divide into 60,10 train and val
#########################################################################################################################################
def extract_validation(train_data):
    """
    Extract every 10th sample from the training data and store it in a validation set.

    Parameters:
        train_data (list or np.array): The training dataset.

    Returns:
        tuple: Updated training dataset and validation dataset.
    """
    # Select every 10th sample for the validation set
    validation_data = train_data[::10]

    # Remove the selected samples from the training dataset
    updated_train_data = [train_data[i] for i in range(len(train_data)) if i % 10 != 0]

    return np.array(updated_train_data), np.array(validation_data)


# Perform the operation for each dataset
celeb_train_hog, celeb_val_hog = extract_validation(celeb_train_hog)
ffhq_train_hog, ffhq_val_hog = extract_validation(ffhq_train_hog)
attgan_train_hog, attgan_val_hog = extract_validation(attgan_train_hog)
stargan_train_hog, stargan_val_hog = extract_validation(stargan_train_hog)
gdwct_train_hog, gdwct_val_hog = extract_validation(gdwct_train_hog)
stylegan2_train_hog, stylegan2_val_hog = extract_validation(stylegan2_train_hog)
stylegan_train_hog, stylegan_val_hog = extract_validation(stylegan_train_hog)

# Print results for verification
print(f"celeb_train: {len(celeb_train_hog)} images, celeb_val: {len(celeb_val_hog)} images")
print(f"ffhq_train: {len(ffhq_train_hog)} images, ffhq_val: {len(ffhq_val_hog)} images")
print(f"attgan_train: {len(attgan_train_hog)} images, attgan_val: {len(attgan_val_hog)} images")
print(f"stargan_train: {len(stargan_train_hog)} images, stargan_val: {len(stargan_val_hog)} images")
print(f"gdwct_train: {len(gdwct_train_hog)} images, gdwct_val: {len(gdwct_val_hog)} images")
print(f"stylegan2_train: {len(stylegan2_train_hog)} images, stylegan2_val: {len(stylegan2_val_hog)} images")
print(f"stylegan_train: {len(stylegan_train_hog)} images, stylegan_val: {len(stylegan_val_hog)} images")
############################################################################################################################################################
#################################################concatenate the labels 0,1 real and fake
#############################################################################################################################################################


celeb_train_labels = np.zeros(len(celeb_train_hog), dtype=int)
ffhq_train_labels = np.zeros(len(ffhq_train_hog), dtype=int)
atta_train_labels = np.ones(len(attgan_train_hog), dtype=int)
star_train_labels = np.ones(len(stargan_train_hog), dtype=int)
gdwct_train_labels = np.ones(len(gdwct_train_hog), dtype=int)
stylegan2_train_labels = np.ones(len(stylegan2_train_hog), dtype=int)
stylegan_train_labels = np.ones(len(stylegan_train_hog), dtype=int)

# Concatenate all training datasets into a single `train` variable
train_hog = np.concatenate([celeb_train_hog, ffhq_train_hog, attgan_train_hog, stargan_train_hog, gdwct_train_hog, stylegan2_train_hog, stylegan_train_hog], axis=0)
train_labels=np.concatenate([celeb_train_labels, ffhq_train_labels, atta_train_labels, star_train_labels, gdwct_train_labels, stylegan2_train_labels,
                              stylegan_train_labels], axis=0)




celeb_test_labels = np.zeros(len(celeb_test_hog), dtype=int)
ffhq_test_labels = np.zeros(len(ffhq_test_hog), dtype=int)
atta_test_labels = np.ones(len(attgan_test_hog), dtype=int)
star_test_labels = np.ones(len(stargan_test_hog), dtype=int)
gdwct_test_labels = np.ones(len(gdwct_test_hog), dtype=int)
stylegan2_test_labels = np.ones(len(stylegan2_test_hog), dtype=int)
stylegan_test_labels = np.ones(len(stylegan_test_hog), dtype=int)

# Concatenate all testing datasets into a single `test` variable
test_hog = np.concatenate([celeb_test_hog, ffhq_test_hog, attgan_test_hog, stargan_test_hog, gdwct_test_hog, stylegan2_test_hog, stylegan_test_hog], axis=0)
test_labels = np.concatenate([celeb_test_labels, ffhq_test_labels, atta_test_labels, star_test_labels, gdwct_test_labels, stylegan2_test_labels,
                        stylegan_test_labels], axis=0)




celeb_val_labels = np.zeros(len(celeb_val_hog), dtype=int)
ffhq_val_labels = np.zeros(len(ffhq_val_hog), dtype=int)
atta_val_labels = np.ones(len(attgan_val_hog), dtype=int)
star_val_labels = np.ones(len(stargan_val_hog), dtype=int)
gdwct_val_labels = np.ones(len(gdwct_val_hog), dtype=int)
stylegan2_val_labels = np.ones(len(stylegan2_val_hog), dtype=int)
stylegan_val_labels = np.ones(len(stylegan_val_hog), dtype=int)

# Concatenate all validation datasets into a single `val` variable
val_hog = np.concatenate([celeb_val_hog, ffhq_val_hog, attgan_val_hog, stargan_val_hog, gdwct_val_hog, stylegan2_val_hog, stylegan_val_hog], axis=0)
val_labels = np.concatenate([celeb_val_labels, ffhq_val_labels, atta_val_labels, star_val_labels, gdwct_val_labels, stylegan2_val_labels,
                       stylegan_val_labels], axis=0)

# Print the results for verification
print(f"Total train: {len(train_hog)} images")
print(f"Total test: {len(test_hog)} images")
print(f"Total val: {len(val_hog)} images")


# Print results for verification
print(f"Train Labels: {len(train_labels)} ")
print(f"Test Labels: {len(test_labels)} ")
print(f"Val Labels: {len(val_labels)} ")



celeb shape: (5000, 224, 224, 3), dtype: uint8
ffhq shape: (5000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
celeb shape: (2500, 160, 160, 3), dtype: uint8
ffhq shape: (2500, 160, 160, 3), dtype: uint8
gdwct shape: (1000, 160, 160, 3), dtype: uint8
attagan shape: (1000, 160, 160, 3), dtype: uint8
stargan shape: (1000, 160, 160, 3), dtype: uint8
stylegan2 shape: (1000, 160, 160, 3), dtype: uint8
stylegan shape: (1000, 160, 160, 3), dtype: uint8
celeb_train: 1750 images, celeb_test: 750 images
ffhq_train: 1750 images, ffhq_test: 750 images
attgan_train: 700 images, attgan_test: 300 images
stargan_train: 700 images, stargan_test: 300 images
gdwct_train: 700 images, gdwct_test: 300 images
stylegan2_train: 700 images, stylegan2_test: 300 images
stylegan_train: 700 images, stylegan

In [8]:
#creating dataloaders
train_dataset = DeepfakeViTDataset(
    train_hog,
    train_labels,
    processor
)

val_dataset = DeepfakeViTDataset(
    val_hog,
    val_labels,
    processor
)

test_dataset = DeepfakeViTDataset(
    test_hog,
    test_labels,
    processor
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,          # Safest setting for Windows/Jupyter
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 394
Validation batches: 44
Testing batches: 188


In [12]:
history = train_vit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS
)

Epoch 01/10 | Train loss: 0.2091 | Train accuracy: 0.9138 | Validation loss: 0.1385 | Validation accuracy: 0.9557
Epoch 02/10 | Train loss: 0.0386 | Train accuracy: 0.9875 | Validation loss: 0.0693 | Validation accuracy: 0.9786
Epoch 03/10 | Train loss: 0.0134 | Train accuracy: 0.9965 | Validation loss: 0.0443 | Validation accuracy: 0.9900
Epoch 04/10 | Train loss: 0.0010 | Train accuracy: 1.0000 | Validation loss: 0.0489 | Validation accuracy: 0.9900
Epoch 05/10 | Train loss: 0.0006 | Train accuracy: 1.0000 | Validation loss: 0.0527 | Validation accuracy: 0.9900
Epoch 06/10 | Train loss: 0.0004 | Train accuracy: 1.0000 | Validation loss: 0.0560 | Validation accuracy: 0.9900
Epoch 07/10 | Train loss: 0.0003 | Train accuracy: 1.0000 | Validation loss: 0.0592 | Validation accuracy: 0.9900
Epoch 08/10 | Train loss: 0.0002 | Train accuracy: 1.0000 | Validation loss: 0.0620 | Validation accuracy: 0.9900
Epoch 09/10 | Train loss: 0.0002 | Train accuracy: 1.0000 | Validation loss: 0.0648 | Va

In [13]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [14]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [15]:
test_results = evaluate_vit_complete(
    model=model,
    test_loader=test_loader,
    device=device
)


VIT-16 TEST RESULTS
test_loss                : 0.103146
accuracy                 : 0.982667
balanced_accuracy        : 0.982667
precision                : 0.984605
recall                   : 0.980667
specificity              : 0.984667
f1_score                 : 0.982632
mcc                      : 0.965341
roc_auc                  : 0.998599
pr_auc                   : 0.998654
average_precision        : 0.998654
eer                      : 0.017667
eer_threshold            : 0.329434
false_positive_rate      : 0.015333
false_negative_rate      : 0.019333

Confusion matrix:
[[1477   23]
 [  29 1471]]

Classification report:
              precision    recall  f1-score   support

        Real     0.9807    0.9847    0.9827      1500
        Fake     0.9846    0.9807    0.9826      1500

    accuracy                         0.9827      3000
   macro avg     0.9827    0.9827    0.9827      3000
weighted avg     0.9827    0.9827    0.9827      3000



In [17]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 17.8%
Time Usage: 130.7 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [18]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 13.4%
Time Usage: 132.9 s
GPU Memory Used: 1653.4 MB
Power Consumption: 93W


save the model

In [19]:
SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_hog_patch16_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "input_size": INPUT_SIZE,
        "label_mapping": {
            0: "real",
            1: "fake"
        }
    },
    os.path.join(
        SAVE_DIR,
        "training_checkpoint.pt"
    )
)

print("ViT-16 celeb model saved:", SAVE_DIR)

ViT-16 celeb model saved: D:\thesis\results\vit_base_hog_patch16_160


#load the model

In [ ]:
import os
import torch

from transformers import (
    ViTImageProcessor,
    ViTForImageClassification
)


SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_hog_patch16_160"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Model directory exists:", os.path.isdir(SAVE_DIR))


# Load the saved processor
processor = ViTImageProcessor.from_pretrained(
    SAVE_DIR
)

# Load the saved model configuration and final trained weights
model = ViTForImageClassification.from_pretrained(
    SAVE_DIR
)

# Move the model to GPU or CPU
model = model.to(device)

# Inference mode
model.eval()


print("ViT-16 model loaded successfully")
print("Number of classes:", model.config.num_labels)
print("Label mapping:", model.config.id2label)

print(
    "Total parameters:",
    sum(parameter.numel() for parameter in model.parameters())
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)

In [ ]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


In [22]:
# ============================================================
# PYTORCH ViT-16 WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# Make sure the loaded model is on the correct device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(test_loader.dataset)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ------------------------------------------------------------
# GPU/model warm-up
# ------------------------------------------------------------

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():
    for batch_index, batch in enumerate(test_loader):

        if batch_index >= WARMUP_BATCHES:
            break

        pixel_values = batch["pixel_values"].to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        outputs = model(
            pixel_values=pixel_values,
            interpolate_pos_encoding=True
        )

        _ = outputs.logits


# Wait until all asynchronous GPU work is complete
if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ------------------------------------------------------------
# Five repeated inference runs
# ------------------------------------------------------------

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):
    print(
        f"\nStarting ViT-16 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous GPU operations remain in the queue
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():
        for batch in test_loader:

            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            outputs = model(
                pixel_values=pixel_values,
                interpolate_pos_encoding=True
            )

            last_logits = outputs.logits

            processed_images += pixel_values.size(0)

    # PyTorch CUDA operations are asynchronous.
    # Synchronize before stopping the timer.
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    # Access one output after synchronization to verify completion
    if last_logits is not None:
        last_output_value = float(
            last_logits[-1, 0]
            .detach()
            .cpu()
            .item()
        )
    else:
        raise RuntimeError(
            "No images were processed during inference."
        )

    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number
    run_summary["processed_images"] = processed_images
    run_summary["last_output_value"] = last_output_value

    run_results.append(run_summary)

    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )

    del last_logits
    del outputs


results_df = pd.DataFrame(run_results)

print("\nIndividual ViT-16 profiling runs:")
display(results_df)

Device: cuda
Test images: 3000
Test batches: 188

Performing warm-up using 3 batches...
Warm-up completed.

Starting ViT-16 resource run 1/5
Run 1: 21.09 seconds | 7.0284 ms/image | 142.28 images/s

Starting ViT-16 resource run 2/5
Run 2: 21.46 seconds | 7.1519 ms/image | 139.82 images/s

Starting ViT-16 resource run 3/5
Run 3: 20.79 seconds | 6.9308 ms/image | 144.28 images/s

Starting ViT-16 resource run 4/5
Run 4: 20.69 seconds | 6.8976 ms/image | 144.98 images/s

Starting ViT-16 resource run 5/5
Run 5: 21.11 seconds | 7.0352 ms/image | 142.14 images/s

Individual ViT-16 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,21.085344,7.028448,142.278919,2.854362,5.825000,4601.593750,4610.175677,4610.453125,8.581927,8.859375,...,78.737634,80.0625,56.155080,100,48.634428,118.209,0.287620,1,3000,-4.64843
1,21.455715,7.151905,139.822883,2.881479,5.825000,4610.894531,4611.353894,4615.699219,0.459363,4.804688,...,78.743455,80.0000,55.968586,100,48.700241,97.071,0.293541,2,3000,-4.64843
2,20.792466,6.930822,144.283027,2.841061,9.350000,4611.503906,4611.690904,4616.316406,0.186998,4.812500,...,78.659218,80.0000,56.731844,100,49.184698,98.756,0.285795,3,3000,-4.64843
3,20.692881,6.897627,144.977398,2.867855,5.328125,4611.558594,4611.683660,4614.062500,0.125066,2.503906,...,78.651685,80.0000,55.786517,100,48.002787,122.517,0.275403,4,3000,-4.64843
4,21.105526,7.035175,142.142865,2.914001,6.718750,4611.589844,4611.724025,4612.019531,0.134181,0.429688,...,78.994350,80.5000,52.949153,100,50.571989,130.522,0.302153,5,3000,-4.64843


In [23]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================

metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("ViT-16 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


ViT-16 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,21.026387,0.300003,20.653884,21.398890
1,latency_ms_per_image,7.008796,0.100001,6.884628,7.132963
2,throughput_images_per_s,142.701019,2.028424,140.182397,145.219640
3,average_cpu_percent,2.871751,0.028014,2.836968,2.906535
4,peak_cpu_percent,6.609375,1.612070,4.607726,8.611024
5,average_ram_mb,4611.325632,0.660212,4610.505870,4612.145393
6,peak_ram_mb,4613.710156,2.466755,4610.647275,4616.773037
7,average_incremental_ram_mb,1.897507,3.739199,-2.745322,6.540336
8,peak_incremental_ram_mb,4.282031,3.142806,0.379722,8.184341
9,average_gpu_memory_mb,3431.561175,0.144157,3431.382180,3431.740170



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 7.009 ± 0.100 (95% CI: 6.885–7.133)
peak_ram_mb: 4613.710 ± 2.467 (95% CI: 4610.647–4616.773)
peak_gpu_memory_mb: 3432.916 ± 0.224 (95% CI: 3432.639–3433.194)
average_gpu_utilization_percent: 55.518 ± 1.479 (95% CI: 53.681–57.355)
average_gpu_power_w: 49.019 ± 0.965 (95% CI: 47.821–50.216)


#genralization

In [25]:
#wild deepfake on dfc
print("\nTest results of DFC on wild deepfake dataset (ViT-16):")
test_dataset = DeepfakeViTDataset(test_images,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)



Test results of DFC on wild deepfake dataset (ViT-16):

VIT-16 TEST RESULTS
test_loss                : 1.872850
accuracy                 : 0.755000
balanced_accuracy        : 0.535556
precision                : 0.763937
recall                   : 0.974444
specificity              : 0.096667
f1_score                 : 0.856445
mcc                      : 0.151233
roc_auc                  : 0.623619
pr_auc                   : 0.820228
average_precision        : 0.820090
eer                      : 0.421037
eer_threshold            : 0.999854
false_positive_rate      : 0.903333
false_negative_rate      : 0.025556

Confusion matrix:
[[  435  4065]
 [  345 13155]]

Classification report:
              precision    recall  f1-score   support

        Real     0.5577    0.0967    0.1648      4500
        Fake     0.7639    0.9744    0.8564     13500

    accuracy                         0.7550     18000
   macro avg     0.6608    0.5356    0.5106     18000
weighted avg     0.7124    0.7550    

In [27]:
#celeb on dfc
print("\nTest results of DFC on Celeb-DF(V2) dataset (ViT-16):")
test_dataset = DeepfakeViTDataset(test_celeb,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)



Test results of DFC on Celeb-DF(V2) dataset (ViT-16):

VIT-16 TEST RESULTS
test_loss                : 0.900679
accuracy                 : 0.884415
balanced_accuracy        : 0.511758
precision                : 0.906309
recall                   : 0.972727
specificity              : 0.050788
f1_score                 : 0.938345
mcc                      : 0.040884
roc_auc                  : 0.519937
pr_auc                   : 0.907602
average_precision        : 0.907730
eer                      : 0.494431
eer_threshold            : 0.999874
false_positive_rate      : 0.949212
false_negative_rate      : 0.027273

Confusion matrix:
[[  29  542]
 [ 147 5243]]

Classification report:
              precision    recall  f1-score   support

        Real     0.1648    0.0508    0.0776       571
        Fake     0.9063    0.9727    0.9383      5390

    accuracy                         0.8844      5961
   macro avg     0.5355    0.5118    0.5080      5961
weighted avg     0.8353    0.8844    0.855

In [29]:
#FF++ on hog
print("\nTest results of dfc on FF++ dataset (ViT-16):")
test_dataset = DeepfakeViTDataset(test_ff,test_ff_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)


Test results of dfc on FF++ dataset (ViT-16):

VIT-16 TEST RESULTS
test_loss                : 4.437746
accuracy                 : 0.445657
balanced_accuracy        : 0.504957
precision                : 0.419385
recall                   : 0.860330
specificity              : 0.149584
f1_score                 : 0.563891
mcc                      : 0.013863
roc_auc                  : 0.528227
pr_auc                   : 0.432736
average_precision        : 0.433175
eer                      : 0.473159
eer_threshold            : 0.999867
false_positive_rate      : 0.850416
false_negative_rate      : 0.139670

Confusion matrix:
[[ 216 1228]
 [ 144  887]]

Classification report:
              precision    recall  f1-score   support

        Real     0.6000    0.1496    0.2395      1444
        Fake     0.4194    0.8603    0.5639      1031

    accuracy                         0.4457      2475
   macro avg     0.5097    0.5050    0.4017      2475
weighted avg     0.5248    0.4457    0.3746      2

# FF++

LOAD THE DATASET

In [7]:
import os, cv2, numpy as np

FINAL_ROOT = r'D:\thesis\ff_final'

def load_split(split, cls):
    base = os.path.join(FINAL_ROOT, split, cls)
    nested, ids = [], []
    for vid_id in sorted(os.listdir(base)):
        d = os.path.join(base, vid_id)
        frames = [cv2.imread(os.path.join(d, f)) for f in sorted(os.listdir(d))]
        if frames:
            nested.append(frames); ids.append(vid_id)
    return nested, ids

# Main splits
ff_real_train_f, ff_real_train_ids = load_split('train', 'real')
ff_fake_train_f, ff_fake_train_ids = load_split('train', 'fake')
ff_real_val_f,   ff_real_val_ids   = load_split('val',   'real')
ff_fake_val_f,   ff_fake_val_ids   = load_split('val',   'fake')
ff_real_test_f,  ff_real_test_ids  = load_split('test',  'real')
ff_fake_test_f,  ff_fake_test_ids  = load_split('test',  'fake')

print("Reloaded main splits. Example shape:", np.shape(ff_real_train_f[0][0]))  # (160,160,3)
print("Real train videos:", len(ff_real_train_f), "| Fake train videos:", len(ff_fake_train_f))
import numpy as np


def combine_split(real_videos, fake_videos):
    """
    Combine all frames from the real and fake video groups.

    Labels:
        0 = Real
        1 = Fake
    """

    real_frames = [
        frame
        for video in real_videos
        for frame in video
        if frame is not None
    ]

    fake_frames = [
        frame
        for video in fake_videos
        for frame in video
        if frame is not None
    ]

    if not real_frames:
        raise ValueError("No real frames found.")

    if not fake_frames:
        raise ValueError("No fake frames found.")

    real_frames = np.stack(real_frames).astype(np.uint8)
    fake_frames = np.stack(fake_frames).astype(np.uint8)

    images = np.concatenate(
        [real_frames, fake_frames],
        axis=0
    )

    real_labels = np.zeros(
        len(real_frames),
        dtype=np.uint8
    )

    fake_labels = np.ones(
        len(fake_frames),
        dtype=np.uint8
    )

    labels = np.concatenate(
        [real_labels, fake_labels],
        axis=0
    )

    return images, labels
# Training data
train_ff, train_ff_labels = combine_split(
    ff_real_train_f,
    ff_fake_train_f
)

# Validation data
val_ff, val_ff_labels = combine_split(
    ff_real_val_f,
    ff_fake_val_f
)

# Testing data
test_ff, test_ff_labels = combine_split(
    ff_real_test_f,
    ff_fake_test_f
)
print("\nTRAIN")
print("Images:", train_ff.shape)
print("Labels:", train_ff_labels.shape)
print("Real:", np.sum(train_ff_labels == 0))
print("Fake:", np.sum(train_ff_labels == 1))

print("\nVALIDATION")
print("Images:", val_ff.shape)
print("Labels:", val_ff_labels.shape)
print("Real:", np.sum(val_ff_labels == 0))
print("Fake:", np.sum(val_ff_labels == 1))

print("\nTEST")
print("Images:", test_ff.shape)
print("Labels:", test_ff_labels.shape)
print("Real:", np.sum(test_ff_labels == 0))
print("Fake:", np.sum(test_ff_labels == 1))

print("\nData types")
print("Train images:", train_ff.dtype)
print("Train labels:", train_ff_labels.dtype)

Reloaded main splits. Example shape: (160, 160, 3)
Real train videos: 517 | Fake train videos: 320

TRAIN
Images: (4595, 160, 160, 3)
Labels: (4595,)
Real: 2948
Fake: 1647

VALIDATION
Images: (948, 160, 160, 3)
Labels: (948,)
Real: 499
Fake: 449

TEST
Images: (2475, 160, 160, 3)
Labels: (2475,)
Real: 1444
Fake: 1031

Data types
Train images: uint8
Train labels: uint8


In [8]:
#creating dataloaders
train_dataset = DeepfakeViTDataset(
    train_ff,
    train_ff_labels,
    processor
)

val_dataset = DeepfakeViTDataset(
    val_ff,
    val_ff_labels,
    processor
)

test_dataset = DeepfakeViTDataset(
    test_ff,
    test_ff_labels,
    processor
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,          # Safest setting for Windows/Jupyter
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 288
Validation batches: 60
Testing batches: 155


In [9]:
history = train_vit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS
)

Epoch 01/10 | Train loss: 0.3125 | Train accuracy: 0.8585 | Validation loss: 0.3854 | Validation accuracy: 0.8344
Epoch 02/10 | Train loss: 0.1903 | Train accuracy: 0.9219 | Validation loss: 0.2643 | Validation accuracy: 0.8797
Epoch 03/10 | Train loss: 0.1456 | Train accuracy: 0.9441 | Validation loss: 0.2377 | Validation accuracy: 0.9114
Epoch 04/10 | Train loss: 0.1091 | Train accuracy: 0.9595 | Validation loss: 0.2414 | Validation accuracy: 0.8892
Epoch 05/10 | Train loss: 0.0920 | Train accuracy: 0.9661 | Validation loss: 0.3093 | Validation accuracy: 0.8776
Epoch 06/10 | Train loss: 0.0664 | Train accuracy: 0.9789 | Validation loss: 0.4569 | Validation accuracy: 0.8565
Epoch 07/10 | Train loss: 0.0605 | Train accuracy: 0.9791 | Validation loss: 0.3671 | Validation accuracy: 0.8629
Epoch 08/10 | Train loss: 0.0564 | Train accuracy: 0.9795 | Validation loss: 0.3390 | Validation accuracy: 0.8924
Epoch 09/10 | Train loss: 0.0660 | Train accuracy: 0.9754 | Validation loss: 0.3058 | Va

In [10]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [11]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [12]:
test_results = evaluate_vit_complete(
    model=model,
    test_loader=test_loader,
    device=device
)


VIT-16 TEST RESULTS
test_loss                : 0.360127
accuracy                 : 0.887677
balanced_accuracy        : 0.886679
precision                : 0.854186
recall                   : 0.880698
specificity              : 0.892659
f1_score                 : 0.867240
mcc                      : 0.770210
roc_auc                  : 0.960203
pr_auc                   : 0.949794
average_precision        : 0.949818
eer                      : 0.114359
eer_threshold            : 0.452125
false_positive_rate      : 0.107341
false_negative_rate      : 0.119302

Confusion matrix:
[[1289  155]
 [ 123  908]]

Classification report:
              precision    recall  f1-score   support

        Real     0.9129    0.8927    0.9027      1444
        Fake     0.8542    0.8807    0.8672      1031

    accuracy                         0.8877      2475
   macro avg     0.8835    0.8867    0.8850      2475
weighted avg     0.8884    0.8877    0.8879      2475



In [13]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 14.7%
Time Usage: 30.6 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [14]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 18.6%
Time Usage: 32.7 s
GPU Memory Used: 1325.1 MB
Power Consumption: 93W


#save the model

In [15]:
SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_ff_patch16_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "input_size": INPUT_SIZE,
        "label_mapping": {
            0: "real",
            1: "fake"
        }
    },
    os.path.join(
        SAVE_DIR,
        "training_checkpoint.pt"
    )
)

print("ViT-16 celeb model saved:", SAVE_DIR)

ViT-16 celeb model saved: D:\thesis\results\vit_base_ff_patch16_160


#load the model

In [ ]:
SAVE_DIR = (
    r"D:\thesis\results"
    r"\vit_base_ff_patch16_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "input_size": INPUT_SIZE,
        "label_mapping": {
            0: "real",
            1: "fake"
        }
    },
    os.path.join(
        SAVE_DIR,
        "training_checkpoint.pt"
    )
)

print("ViT-16 celeb model saved:", SAVE_DIR)

Model saved successfully:
D:\thesis\results\Inception_V3_ff_10epochs.h5


In [16]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_24800\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [17]:
# ============================================================
# PYTORCH ViT-16 WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# Make sure the loaded model is on the correct device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(test_loader.dataset)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ------------------------------------------------------------
# GPU/model warm-up
# ------------------------------------------------------------

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():
    for batch_index, batch in enumerate(test_loader):

        if batch_index >= WARMUP_BATCHES:
            break

        pixel_values = batch["pixel_values"].to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        outputs = model(
            pixel_values=pixel_values,
            interpolate_pos_encoding=True
        )

        _ = outputs.logits


# Wait until all asynchronous GPU work is complete
if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ------------------------------------------------------------
# Five repeated inference runs
# ------------------------------------------------------------

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):
    print(
        f"\nStarting ViT-16 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous GPU operations remain in the queue
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():
        for batch in test_loader:

            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            outputs = model(
                pixel_values=pixel_values,
                interpolate_pos_encoding=True
            )

            last_logits = outputs.logits

            processed_images += pixel_values.size(0)

    # PyTorch CUDA operations are asynchronous.
    # Synchronize before stopping the timer.
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    # Access one output after synchronization to verify completion
    if last_logits is not None:
        last_output_value = float(
            last_logits[-1, 0]
            .detach()
            .cpu()
            .item()
        )
    else:
        raise RuntimeError(
            "No images were processed during inference."
        )

    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number
    run_summary["processed_images"] = processed_images
    run_summary["last_output_value"] = last_output_value

    run_results.append(run_summary)

    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )

    del last_logits
    del outputs


results_df = pd.DataFrame(run_results)

print("\nIndividual ViT-16 profiling runs:")
display(results_df)

Device: cuda
Test images: 2475
Test batches: 155

Performing warm-up using 3 batches...
Warm-up completed.

Starting ViT-16 resource run 1/5
Run 1: 17.79 seconds | 7.1866 ms/image | 139.15 images/s

Starting ViT-16 resource run 2/5
Run 2: 17.77 seconds | 7.1808 ms/image | 139.26 images/s

Starting ViT-16 resource run 3/5
Run 3: 17.81 seconds | 7.1947 ms/image | 138.99 images/s

Starting ViT-16 resource run 4/5
Run 4: 17.66 seconds | 7.1339 ms/image | 140.18 images/s

Starting ViT-16 resource run 5/5
Run 5: 18.51 seconds | 7.4800 ms/image | 133.69 images/s

Individual ViT-16 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,17.786758,7.186569,139.148458,2.869634,5.825000,5757.296875,5758.141541,5758.972656,0.844666,1.675781,...,97.931034,100.0,47.972414,100,50.571193,128.205,0.255690,1,2475,-1.111867
1,17.772478,7.180799,139.260263,2.912042,7.103125,5758.882812,5759.035182,5762.496094,0.152370,3.613281,...,98.000000,100.0,48.100000,100,50.179827,128.475,0.252317,2,2475,-1.111867
2,17.806948,7.194727,138.990687,2.921854,6.215625,5758.898438,5759.056110,5761.257812,0.157673,2.359375,...,98.013245,100.0,47.721854,100,51.675967,128.711,0.259832,3,2475,-1.111867
3,17.656453,7.133921,140.175376,2.914828,7.103125,5758.949219,5759.120255,5763.750000,0.171036,4.800781,...,97.718121,100.0,48.093960,100,50.098047,133.045,0.251050,4,2475,-1.111867
4,18.512932,7.479973,133.690331,2.944531,5.375000,5758.960938,5759.103265,5763.640625,0.142328,4.679688,...,98.076923,100.0,43.794872,100,53.008301,132.757,0.274499,5,2475,-1.111867


In [19]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("ViT-16 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


ViT-16 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,17.907114,0.343690,17.480367,18.333862
1,latency_ms_per_image,7.235198,0.138865,7.062774,7.407621
2,throughput_images_per_s,138.253023,2.592011,135.034616,141.471430
3,average_cpu_percent,2.912578,0.027187,2.878820,2.946335
4,peak_cpu_percent,6.324375,0.770620,5.367524,7.281226
5,average_ram_mb,5758.891271,0.420521,5758.369125,5759.413417
6,peak_ram_mb,5762.023438,1.981992,5759.562470,5764.484405
7,average_incremental_ram_mb,0.293614,0.308221,-0.089093,0.676322
8,peak_incremental_ram_mb,3.425781,1.387244,1.703290,5.148272
9,average_gpu_memory_mb,3434.154896,0.138487,3433.982942,3434.326850



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 7.235 ± 0.139 (95% CI: 7.063–7.408)
peak_ram_mb: 5762.023 ± 1.982 (95% CI: 5759.562–5764.484)
peak_gpu_memory_mb: 3436.207 ± 0.000 (95% CI: 3436.207–3436.207)
average_gpu_utilization_percent: 47.137 ± 1.874 (95% CI: 44.809–49.464)
average_gpu_power_w: 51.107 ± 1.235 (95% CI: 49.573–52.640)


#gernalization

In [21]:
#wild deepfake on ff
print("\nTest results of FF++ on wild deepfake dataset (ViT-16):")
test_dataset = DeepfakeViTDataset(test_images,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)



Test results of FF++ on wild deepfake dataset (ViT-16):

VIT-16 TEST RESULTS
test_loss                : 2.261549
accuracy                 : 0.382611
balanced_accuracy        : 0.497815
precision                : 0.746948
recall                   : 0.267407
specificity              : 0.728222
f1_score                 : 0.393825
mcc                      : -0.004270
roc_auc                  : 0.466116
pr_auc                   : 0.742028
average_precision        : 0.742066
eer                      : 0.525296
eer_threshold            : 0.098357
false_positive_rate      : 0.271778
false_negative_rate      : 0.732593

Confusion matrix:
[[3277 1223]
 [9890 3610]]

Classification report:
              precision    recall  f1-score   support

        Real     0.2489    0.7282    0.3710      4500
        Fake     0.7469    0.2674    0.3938     13500

    accuracy                         0.3826     18000
   macro avg     0.4979    0.4978    0.3824     18000
weighted avg     0.6224    0.3826    0.

In [23]:
#celeb on ff
print("\nTest results of FF++ on Celeb-df(v2) dataset (ViT-16):")
test_dataset = DeepfakeViTDataset(test_celeb,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)



Test results of FF++ on Celeb-df(v2) dataset (ViT-16):

VIT-16 TEST RESULTS
test_loss                : 3.512851
accuracy                 : 0.235531
balanced_accuracy        : 0.569444
precision                : 0.988277
recall                   : 0.156401
specificity              : 0.982487
f1_score                 : 0.270062
mcc                      : 0.116728
roc_auc                  : 0.710066
pr_auc                   : 0.955993
average_precision        : 0.956001
eer                      : 0.353877
eer_threshold            : 0.004748
false_positive_rate      : 0.017513
false_negative_rate      : 0.843599

Confusion matrix:
[[ 561   10]
 [4547  843]]

Classification report:
              precision    recall  f1-score   support

        Real     0.1098    0.9825    0.1976       571
        Fake     0.9883    0.1564    0.2701      5390

    accuracy                         0.2355      5961
   macro avg     0.5491    0.5694    0.2338      5961
weighted avg     0.9041    0.2355    0.26

In [25]:
#DFC on ff
print("\nTest results of FF++ on DFC dataset (ViT-16):")
test_dataset = DeepfakeViTDataset(test_hog,test_labels,processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_results = evaluate_vit_complete(model=model,test_loader=test_loader,device=device)


Test results of FF++ on DFC dataset (ViT-16):

VIT-16 TEST RESULTS
test_loss                : 1.658813
accuracy                 : 0.411667
balanced_accuracy        : 0.411667
precision                : 0.383670
recall                   : 0.291333
specificity              : 0.532000
f1_score                 : 0.331186
mcc                      : -0.182017
roc_auc                  : 0.369167
pr_auc                   : 0.425114
average_precision        : 0.425761
eer                      : 0.593667
eer_threshold            : 0.312110
false_positive_rate      : 0.468000
false_negative_rate      : 0.708667

Confusion matrix:
[[ 798  702]
 [1063  437]]

Classification report:
              precision    recall  f1-score   support

        Real     0.4288    0.5320    0.4749      1500
        Fake     0.3837    0.2913    0.3312      1500

    accuracy                         0.4117      3000
   macro avg     0.4062    0.4117    0.4030      3000
weighted avg     0.4062    0.4117    0.4030      